# 08. Multi-Agent Patterns


## Learning Objectives

Understand and implement five multi-agent patterns.

This notebook covers:
- **Subagents**: the main agent calls specialized subagents as tools
- **Handoffs**: state transitions between agents with `Command(goto=...)`
- **Skills**: one agent loads specialized prompts depending on the task
- **Router**: a classifier routes input to the right agent
- **Custom**: developer-controlled complex workflows


## 8.1 Environment Setup


In [ ]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)

from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-4.1",
)

print("모델 준비 완료:", model.model_name)

In [ ]:
# Optional observability setup: LangSmith or Langfuse
# Set the keys in .env, or uncomment the lines below to enter them manually.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: automatically enabled when LANGSMITH_TRACING=true
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: pass config={"callbacks": [langfuse_handler]} to invoke/stream
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


## 8.2 Comparing Multi-Agent Patterns

The table below compares five multi-agent patterns. Each one fits a different situation, so you should choose based on your project requirements.

| Pattern | Routing Owner | State Sharing | Best Fit |
|------|-----------|----------|------------|
| **Subagents** | Main agent | Isolated through tools | Parallel work, distributed execution |
| **Handoffs** | Tool call | State transition | Sequential multi-hop flows |
| **Skills** | Single agent | Prompt swapping | Domain specialization |
| **Router** | Classifier | Parallel execution | Multi-domain systems |
| **Custom** | Developer-defined | Full control | Complex workflows |

### Key differences
- **Subagents** run independently and return only the result
- **Handoffs** pass conversation state between agents
- **Skills** let one agent switch roles
- **Router** classifies input and delegates it to the right specialist


## 8.3 The Subagent Pattern

In this pattern, the main agent (supervisor) calls specialized subagents **as tools**.

### Characteristics
- Each subagent is wrapped in a tool function
- The main agent decides which subagent to call
- The internal state of each subagent is isolated from the main agent
- Parallel execution is possible, which can improve performance


In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool

# 전문 도구 정의
@tool
def math_expert(question: str) -> str:
    """수학 문제를 풀어줍니다. 수학적 계산이 필요할 때 사용하세요."""
    # 실제로는 계산 로직이 들어갑니다
    return f"수학 답변: '{question}'에 대한 답이 계산되었습니다."

@tool
def code_expert(question: str) -> str:
    """프로그래밍 질문에 답합니다. 코딩 관련 질문에 사용하세요."""
    return f"코드 답변: '{question}'에 대한 솔루션입니다"

@tool
def general_search(query: str) -> str:
    """일반 정보를 검색합니다."""
    return f"'{query}'에 대한 검색 결과"

# 감독자(supervisor) 에이전트
supervisor = create_agent(
    model=model,
    tools=[math_expert, code_expert, general_search],
    system_prompt="""당신은 전문가에게 작업을 위임하는 감독 에이전트입니다:
- 수학 문제는 math_expert를 사용하세요
- 프로그래밍 질문은 code_expert를 사용하세요
- 그 외에는 general_search를 사용하세요
항상 가장 적절한 전문가에게 위임하세요.""",
)

result = supervisor.invoke(
    {"messages": [{"role": "user", "content": "10의 팩토리얼은 얼마인가요?"}]},
    config=lf_config,
)
print("서브에이전트 결과:", result["messages"][-1].content)

## 8.4 The Handoff Pattern

This pattern uses `Command(goto=...)` to **transfer state** between agents.

### Characteristics
- A tool returns a `Command` object that routes execution to another agent
- Conversation state (message history) is passed to the next agent
- A `StateGraph` defines the flow between agents
- This fits multi-hop scenarios such as customer-service transfers


In [ ]:
from langgraph.types import Command
from langgraph.graph import StateGraph, START, END, MessagesState

# 핸드오프를 위한 도구
@tool
def transfer_to_sales() -> Command:
    """대화를 영업 부서로 전환합니다."""
    return Command(goto="sales_agent", graph=Command.PARENT)

@tool
def transfer_to_support() -> Command:
    """대화를 기술 지원 부서로 전환합니다."""
    return Command(goto="support_agent", graph=Command.PARENT)

@tool
def resolve_query(answer: str) -> str:
    """사용자에게 최종 답변을 제공합니다."""
    return answer

# 라우터 에이전트
router_agent = create_agent(
    model=model,
    tools=[transfer_to_sales, transfer_to_support],
    system_prompt="당신은 안내 데스크입니다. 고객을 적절한 부서로 안내하세요.",
    name="router",
)

# 영업 에이전트
sales_agent = create_agent(
    model=model,
    tools=[resolve_query],
    system_prompt="당신은 영업 에이전트입니다. 가격 및 제품 문의를 도와주세요.",
    name="sales_agent",
)

# 지원 에이전트
support_agent = create_agent(
    model=model,
    tools=[resolve_query],
    system_prompt="당신은 기술 지원 에이전트입니다. 기술적 문제를 도와주세요.",
    name="support_agent",
)

# 핸드오프 그래프 구성
builder = StateGraph(MessagesState)
builder.add_node(router_agent)
builder.add_node(sales_agent)
builder.add_node(support_agent)
builder.add_edge(START, "router")

graph = builder.compile()

result = graph.invoke(
    {"messages": [{"role": "user", "content": "엔터프라이즈 플랜 가격을 알고 싶습니다."}]},
    config=lf_config,
)
print("핸드오프 결과:", result["messages"][-1].content)

## 8.5 The Skill Pattern

A single agent dynamically **loads a specialized prompt** depending on the task.

### Characteristics
- One agent has multiple skills
- Each skill is implemented as a specialized system prompt
- The agent dynamically loads the skill it needs
- One agent can handle many tasks without managing multiple separate agents


In [ ]:
# 스킬 정의
skills = {
    "translator": "당신은 전문 번역가입니다. 언어 간 텍스트를 정확하게 번역하세요.",
    "summarizer": "당신은 전문 요약가입니다. 긴 텍스트를 간결하게 요약하세요.",
    "coder": "당신은 전문 프로그래머입니다. 깔끔하고 효율적인 코드를 작성하세요.",
}

@tool
def load_skill(skill_name: str) -> str:
    """전문 스킬을 로드합니다. 사용 가능한 스킬: translator, summarizer, coder."""
    if skill_name in skills:
        return f"스킬 로드 완료: {skill_name}. 지침: {skills[skill_name]}"
    return f"알 수 없는 스킬: {skill_name}. 사용 가능: {list(skills.keys())}"

skill_agent = create_agent(
    model=model,
    tools=[load_skill],
    system_prompt="""당신은 전문 기술에 접근할 수 있는 다재다능한 어시스턴트입니다.
사용자의 요청을 처리하기 전에 적절한 스킬을 로드하세요.""",
)

result = skill_agent.invoke(
    {"messages": [{"role": "user", "content": "'Hello World'를 한국어와 일본어로 번역해 주세요."}]},
    config=lf_config,
)
print("스킬 패턴 결과:", result["messages"][-1].content)

## 8.6 The Router Pattern

A classifier **routes** input to the most appropriate agent.

### Characteristics
- The query is classified first
- It is then delegated to the right specialist agent or tool
- This is useful in multi-domain systems
- Routing logic can be rule-based or model-based


In [ ]:
from langgraph.types import Send

@tool
def classify_query(query: str) -> str:
    """쿼리를 카테고리로 분류합니다: math, code, general."""
    query_lower = query.lower()
    if any(w in query_lower for w in ["calculate", "math", "sum", "multiply"]):
        return "math"
    elif any(w in query_lower for w in ["code", "program", "function", "python"]):
        return "code"
    return "general"

# 라우터: 분류 결과에 따라 전문 에이전트로 전달
router = create_agent(
    model=model,
    tools=[classify_query, math_expert, code_expert, general_search],
    system_prompt="""당신은 라우팅 에이전트입니다. 먼저 쿼리를 분류한 다음 적절한 전문가에게 전달하세요:
- 수학 쿼리 -> math_expert 사용
- 코드 쿼리 -> code_expert 사용
- 일반 쿼리 -> general_search 사용""",
)

result = router.invoke(
    {"messages": [{"role": "user", "content": "리스트를 정렬하는 Python 함수를 작성해 주세요."}]},
    config=lf_config,
)
print("라우터 결과:", result["messages"][-1].content)

## 8.7 Choosing a Pattern

Which multi-agent pattern should you choose? Use the guide below.

### Decision tree

1. **Can the agents work independently?**
   - YES → **Subagents** (parallel execution, result aggregation)
   - NO → move to the next question

2. **Must conversation state be passed between agents?**
   - YES → **Handoffs** (state transition, multi-hop)
   - NO → move to the next question

3. **Can a single agent just switch roles?**
   - YES → **Skills** (prompt switching)
   - NO → move to the next question

4. **Is classifying input and sending it to a handler enough?**
   - YES → **Router** (classify and delegate)
   - NO → **Custom** (fully custom graph)

### Practical guidance
- Start with the **simplest pattern** (usually Subagents or Skills)
- Move to Handoffs or Router only when requirements become more complex
- Use a Custom pattern only when the other patterns are not enough
- You can also **combine** patterns (for example, Router + Handoffs)


## 8.8 Summary

| Pattern | Core API | When to use it |
|------|---------|----------|
| **Subagents** | `create_agent` + tool functions | Independent parallel work |
| **Handoffs** | `Command(goto=...)`, `StateGraph` | Multi-hop state transfer |
| **Skills** | Load prompts as tools | One agent with many roles |
| **Router** | Classifier tool + specialist tools | Multi-domain classification |
| **Custom** | Full `StateGraph` control | Complex business logic |

### Key principles
- Start simple and increase complexity only when necessary
- Design each agent around **one clear responsibility**
- Define the **interfaces** between agents (inputs and outputs) clearly

### Next Steps
→ **[09_custom_workflow_and_rag.ipynb](./09_custom_workflow_and_rag.ipynb)**: Learn about custom workflows and RAG.
